# L6 Appendix. Cloud Sync: One Memory, Many Devices (optional)

Everything in L6 runs with no network. This appendix is the one path memory takes off the device, and it is off by default: push your assistant's memory to your own Qdrant Cloud cluster, then watch a second device pull it down and know everything the first one learned. Run L6 first so `assistant_shard` exists.

## 1. Push this device's memory to the cloud

Syncing is a choice you make, never a default. Set `USE_CLOUD` to `True` and put your cluster's URL and API key in the `QDRANT_URL` and `QDRANT_API_KEY` environment variables.

In [1]:
import os
from qdrant_edge import EdgeShard, ScrollRequest

USE_CLOUD = False

client = None
have_keys = os.getenv("QDRANT_URL") and os.getenv("QDRANT_API_KEY")
if USE_CLOUD and have_keys:
    from qdrant_client import QdrantClient, models
    client = QdrantClient(url=os.environ["QDRANT_URL"],
                          api_key=os.environ["QDRANT_API_KEY"])

if client is None:
    print("Nothing uploaded. Every memory stays on this device.")
elif client.collection_exists("assistant_memory"):
    client = None
    print("assistant_memory already exists on the cluster.",
          "Delete it there first, or rename the collection here.")
else:
    assistant_shard = EdgeShard.load("./assistant_shard")
    records, _ = assistant_shard.scroll(
        ScrollRequest(limit=1000,
                      with_payload=True,
                      with_vector=True)
    )
    client.create_collection(
        "assistant_memory",
        vectors_config={
            "text": models.VectorParams(
                size=768, distance=models.Distance.COSINE),
            "image": models.VectorParams(
                size=512, distance=models.Distance.COSINE),
        },
    )
    client.upsert("assistant_memory", points=[
        models.PointStruct(id=r.id, vector=r.vector,
                           payload=r.payload)
        for r in records
    ])
    assistant_shard.close()
    print(f"Pushed {len(records)} memories: same points, same",
          "format, readable by any Qdrant server or device.")

Nothing uploaded. Every memory stays on this device.


## 2. A second device pulls it down

A brand-new device has never seen your day and never taught anything. It downloads one shard snapshot from the cluster, unpacks it, and knows both. In the notebook, "device B" is just a second folder.

In [2]:
from helper import fetch_snapshot, fresh_start, text_search

if client is None:
    print("No cluster configured. Every memory stays on this device.")
else:
    DEVICE_B = fresh_start("./device_b_shard")
    snapshot = fetch_snapshot(os.environ["QDRANT_URL"],
                              os.environ["QDRANT_API_KEY"],
                              "assistant_memory",
                              "./assistant.snapshot")
    EdgeShard.unpack_snapshot(snapshot, DEVICE_B)
    device_b = EdgeShard.load(DEVICE_B)

    hit = text_search(device_b, "the ramen place downtown", limit=1)[0]
    print(f"Device B recalls your day: {hit.score:.3f} ",
          hit.payload.get("note") or hit.payload["transcript"])
    taught = device_b.retrieve([5000], True, False)[0]
    print("Device B knows what you taught:", taught.payload["note"])

No cluster configured. Every memory stays on this device.


## 3. Stay in sync: pull only what's new

Another device in the fleet stores a memory on the cluster. Yours sends its snapshot manifest, and the server answers with a partial snapshot holding only the changes.

In [3]:
from helper import embed_text

if client is None:
    print("No cluster configured. Every memory stays on this device.")
else:
    note = "The spare batteries are in the charging dock by the gate"
    client.upsert("assistant_memory", points=[models.PointStruct(
        id=6000,
        vector={"text": embed_text([note])[0]},
        payload={"source_type": "text", "note": note},
    )])

    manifest = device_b.snapshot_manifest()
    partial = fetch_snapshot(os.environ["QDRANT_URL"],
                             os.environ["QDRANT_API_KEY"],
                             "assistant_memory",
                             "./partial.snapshot",
                             manifest=manifest)
    device_b.update_from_snapshot(partial)

    hit = text_search(device_b, "where are the spare batteries?",
                      limit=1)[0]
    print(f"Device B learned it without being taught: {hit.score:.3f} ",
          hit.payload["note"])
    device_b.close()

No cluster configured. Every memory stays on this device.


## Where this goes

This push-pull loop is the pattern behind fleet memory: every device pushes what it learns, every device pulls what the fleet knows. The production version keeps two shards per device (a mutable one for local writes, a mirrored one refreshed by partial snapshots) and is documented in the [Edge synchronization guide](https://qdrant.tech/documentation/edge/edge-synchronization-guide/), with a reference implementation in [qdrant-edge-demo](https://github.com/qdrant/qdrant-edge-demo). [memory-fleet](https://github.com/qdrant-labs/memory-fleet) runs it on a real robot fleet: what one robot learns, the whole fleet remembers, which is exactly where Lesson 6's robot is headed.